# CW network - a NEW network from scratch, with the real recordings in from the start (2026-11-19)

Run by Claude. Output: My Drive folder **cw_scratch_out**. Resumable: if Colab stops, run boxes 1, 2 and 4 again.

In [ ]:
# 1. Connect Google Drive and unpack the files
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
os.makedirs('/content/cw', exist_ok=True)
D = '/content/drive/MyDrive/'
zipfile.ZipFile(D + 'cw_real.zip').extractall('/content/cw')          # practice data, noise bank, W1AW sound
for z in ['cw_code.zip', 'cw_own3.zip']:                                # newer code; his three on-air sessions
    if os.path.exists(D + z):
        zipfile.ZipFile(D + z).extractall('/content/cw')
zipfile.ZipFile(D + 'cw_scratch.zip').extractall('/content/cw')       # LAST: this run's code and the labels with notes
print(sorted(os.listdir('/content/cw/deepnet')), sorted(os.listdir('/content/cw/data')))

In [ ]:
# 2. Is the GPU there?
import torch, os
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('CPU cores:', os.cpu_count())

In [ ]:
# 3. A short test (a few minutes): it should print steps and HELD-OUT lines
!python /content/cw/deepnet/finetune_gpu.py --own /content/cw/data/own_cut3 --w1aw /content/cw/data/w1aw_cut --scratch base --out /content/test_out --steps 50 --workers 2 --centred 2>&1 | grep -v -i warn | tail -6

In [ ]:
# 4. THE TRAINING, from scratch. Saves every 250 steps into Drive (cw_scratch_out); run again and it goes on.
!python /content/cw/deepnet/finetune_gpu.py --own /content/cw/data/own_cut3 --w1aw /content/cw/data/w1aw_cut --scratch base --out /content/drive/MyDrive/cw_scratch_out --steps 40000 --workers 2 --centred 2>&1 | grep -v -i warn

In [ ]:
# 5. How it is doing (held-out error every 250 steps - lower is better)
!grep HELD /content/drive/MyDrive/cw_scratch_out/train.log | tail -12